**Imports and environment setup**

In [1]:
# imports

import os
import requests
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display
from openai import OpenAI

# If you get an error running this cell, then please head over to the troubleshooting notebook!

**Load and check API keys**

In [2]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

load_dotenv(override=True)

google_api_key = os.getenv("GOOGLE_API_KEY")

if not google_api_key:
    print("No API key was found - please be sure to add your key to the .env file, and save the file! Or you can skip the next 2 cells if you don't want to use Gemini")
elif not google_api_key.startswith("AIz"):
    print("An API key was found, but it doesn't start AIz")
else:
    print("API key found and looks good so far!")

API key found and looks good so far!


**Initialize Gemini client**

In [9]:
#  Set up the Gemini client
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

gemini = OpenAI(
    base_url=GEMINI_BASE_URL,
    api_key=google_api_key
)

**System prompt for football standings**

In [3]:
system_prompt = """You are a professional sports data assistant.

Your task is to present football league standings in a clean, well-structured, and visually appealing format.

Rules:
- Always format the standings as a neat table.
- Include columns: Position, Team, Played, Wins, Draws, Losses, Goals For, Goals Against, Goal Difference, Points.
- Add emojis to make the output visually engaging (e.g., 🥇 🥈 🥉 ⚽).
- Highlight the top 3 teams.
- Keep the formatting consistent and easy to read.
- Do not invent data — only use the provided input.
- If data is incomplete, present what is available clearly.
- no news
-Add statistics for the league
Optional:
- Add a short summary of the top team performance."""

**User prompt for football standings**

In [4]:
# A function that writes a User Prompt that asks for summaries of websites:

def user_prompt_for(website):
    
    user_prompt = f"""Here are football league standings for multiple leagues:

{website.text}

Please:
- Separate each league clearly
- Display each league in a clean table in english
-show the whole table
- Make the output visually appealing and easy to read.
-display results without writing any other text
- display thr related match and news and other for the leauge on
-dont write any info about the league except the standings and next matches(table)
-put next matches in a table""" 
    return user_prompt

In [5]:
# See how this function creates exactly the format above

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(website)}
    ]

**Install Selenium (for JS-heavy websites)**

In [ ]:
!pip install selenium

In [ ]:
!pip install webdriver-manager

**ScrapeWebsite class**

In [6]:
# A class to represent a Webpage
# If you're not familiar with Classes, check out the "Intermediate Python" notebook

# Some websites need you to use proper headers when fetching them:
# Import necessary modules
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import time

class ScrapeWebsite:
    def __init__(self, url):
        """
        Create this Website object from the given URL using Selenium + BeautifulSoup
        Supports JavaScript-heavy and normal websites uniformly.
        """
        self.url = url

        # Configure headless Chrome
        options = Options()
        options.add_argument('--headless')
        options.add_argument('--no-sandbox')
        options.add_argument('--disable-dev-shm-usage')

        # Use webdriver-manager to manage ChromeDriver
        service = Service(ChromeDriverManager().install())

        # Initialize the Chrome WebDriver with the service and options
        driver = webdriver.Chrome(service=service, options=options)

        # Start Selenium WebDriver
        driver.get(url)

        # Wait for JS to load (adjust as needed)
        time.sleep(3)

        # Fetch the page source after JS execution
        page_source = driver.page_source
        driver.quit()

        # Parse the HTML content with BeautifulSoup
        soup = BeautifulSoup(page_source, 'html.parser')

        # Extract title
        self.title = soup.title.string if soup.title else "No title found"

        # Remove unnecessary elements
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()

        # Extract the main text
        self.text = soup.body.get_text(separator="\n", strip=True)


**Summarization function**

In [7]:
def summarize_js_website(url):
    website = ScrapeWebsite(url)
    response = gemini.chat.completions.create(
        model = "gemini-2.5-flash",
        messages = messages_for(website)
    )
    return response.choices[0].message.content

In [10]:


summary=summarize_js_website("https://www.yallakora.com/tour-standing/93/%d8%a7%d9%84%d8%af%d9%88%d8%b1%d9%8a-%d8%a7%d9%84%d8%a5%d9%86%d8%ac%d9%84%d9%8a%d8%b2%d9%8a#tour-groupstandingclip ")

In [11]:
display(Markdown(summary))

## English Premier League 🏴󠁧󠁢󠁥󠁮󠁧󠁿

### Standings

| Position | Team                 | Played | Wins | Draws | Losses | Goals For ⚽ | Goals Against | Goal Difference | Points |
| :------- | :------------------- | :----- | :--- | :---- | :----- | :----------- | :------------ | :-------------- | :----- |
| 🥇       | Arsenal              | 31     | 21   | 7     | 3      | 61           | 22            | 39              | 70     |
| 🥈       | Manchester City      | 30     | 18   | 7     | 5      | 60           | 28            | 32              | 61     |
| 🥉       | Manchester United    | 31     | 15   | 10    | 6      | 56           | 43            | 13              | 55     |
| 4        | Aston Villa          | 31     | 16   | 6     | 9      | 42           | 37            | 5               | 54     |
| 5        | Liverpool            | 31     | 14   | 7     | 10     | 50           | 42            | 8               | 49     |
| 6        | Chelsea              | 31     | 13   | 9     | 9      | 53           | 38            | 15              | 48     |
| 7        | Brentford            | 31     | 13   | 7     | 11     | 46           | 42            | 4               | 46     |
| 8        | Everton              | 31     | 13   | 7     | 11     | 37           | 35            | 2               | 46     |
| 9        | Fulham               | 31     | 13   | 5     | 13     | 43           | 44            | -1              | 44     |
| 10       | Brighton             | 31     | 11   | 10    | 10     | 41           | 37            | 4               | 43     |
| 11       | Sunderland           | 31     | 11   | 10    | 10     | 32           | 36            | -4              | 43     |
| 12       | Newcastle            | 31     | 12   | 6     | 13     | 44           | 45            | -1              | 42     |
| 13       | Bournemouth          | 31     | 9    | 15    | 7      | 46           | 48            | -2              | 42     |
| 14       | Crystal Palace       | 30     | 10   | 9     | 11     | 33           | 35            | -2              | 39     |
| 15       | Leeds United         | 31     | 7    | 12    | 12     | 37           | 48            | -11             | 33     |
| 16       | Nottingham Forest    | 31     | 8    | 8     | 15     | 31           | 43            | -12             | 32     |
| 17       | Tottenham Hotspur    | 31     | 7    | 9     | 15     | 40           | 50            | -10             | 30     |
| 18       | West Ham United      | 31     | 7    | 8     | 16     | 36           | 57            | -21             | 29     |
| 19       | Burnley              | 31     | 4    | 8     | 19     | 33           | 61            | -28             | 20     |
| 20       | Wolverhampton        | 31     | 3    | 8     | 20     | 24           | 54            | -30             | 17     |

### League Statistics

*   **Matches Played:** 309 out of 380
*   **Top Scorer:** Erling Haaland (Manchester City) - 22 goals ⚽
*   **Top Assister:** Bruno Fernandes (Manchester United) - 16 assists 🎯

### Next Matches (Week 32)

| Match                               | Date & Time (Local)             |
| :---------------------------------- | :------------------------------ |
| West Ham United vs Wolverhampton    | Friday, April 10, 2026 - 21:00  |
| Arsenal vs Bournemouth              | Saturday, April 11, 2026 - 13:30 |
| Brentford vs Everton                | Saturday, April 11, 2026 - 16:00 |